In [1]:
#!/usr/bin/env python3
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 5.1. Machine Learning — Regresión
#        (validación interna con un único split temporal 80/20)
# ===================================================================

import pandas as pd
import numpy as np
import os
import pickle
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_DIR  = 'ml_normalized_grouped'
OUTPUT_DIR = 'ml_results_grouped'
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLASSROOM = '1.6'

# ── Límites de muestras de entrenamiento ────────────────────────────
# Basados en análisis de learning curves (Paso 6 — Gráfica 8)
# Modelos sin entrada aquí usan todos los datos de train disponibles
TRAIN_LIMIT = {
    'KNN (k=11)':        29,
    'Gradient Boosting': 29,   # Limitar muestras para evitar overfitting severo
}

# ==========================================
# 1. LOADING DATA
# ==========================================
print("\n" + "="*70)
print("AULA 1.6 - MACHINE LEARNING REGRESIÓN")
print("="*70)
print("\n1. LOADING DATA...")

try:
    X_train_norm = pd.read_excel(os.path.join(INPUT_DIR, 'X_train_normalised.xlsx')).values
    X_test_norm  = pd.read_excel(os.path.join(INPUT_DIR, 'X_test_normalised.xlsx')).values
    X_train_raw  = pd.read_excel(os.path.join(INPUT_DIR, 'X_train.xlsx')).values
    X_test_raw   = pd.read_excel(os.path.join(INPUT_DIR, 'X_test.xlsx')).values
    y_train = pd.read_excel(os.path.join(INPUT_DIR, 'y_train.xlsx')).values.ravel()
    y_test  = pd.read_excel(os.path.join(INPUT_DIR, 'y_test.xlsx')).values.ravel()

    print(f"   Train: {len(y_train)}, Test: {len(y_test)}")
    print(f"   y_train: min={y_train.min():.0f}, max={y_train.max():.0f}, mean={y_train.mean():.2f}")
    print(f"   y_test:  min={y_test.min():.0f},  max={y_test.max():.0f},  mean={y_test.mean():.2f}")

    # ===== NUEVO: Desviación típica de y (referencia para umbrales) =====
    STD_Y = y_train.std()
    print(f"   STD_Y (referencia para diagnóstico): {STD_Y:.2f} personas")

except FileNotFoundError as e:
    print(f"   ERROR: {e}")
    exit(1)

# ==========================================
# 2. DEFINIR MODELOS
# ==========================================
print("\n2. DEFINING MODELS...")

needs_scaling = {
    'Linear Regression', 'Ridge', 'Lasso', 'ElasticNet',
    'SVR (rbf)', 'SVR (linear)', 'SVR (poly deg2)',
    'KNN (k=3)', 'KNN (k=5)', 'KNN (k=7)', 'KNN (k=9)', 'KNN (k=11)'
}

models = {
    'Linear Regression': LinearRegression(),
    'Ridge':             Ridge(alpha=10.0, random_state=42),
    'Lasso':             Lasso(alpha=0.5, random_state=42),
    'ElasticNet':        ElasticNet(alpha=0.5, l1_ratio=0.5, random_state=42),

    'Decision Tree':     DecisionTreeRegressor(
                             max_depth=5, min_samples_split=5,
                             min_samples_leaf=3, random_state=42),
    'Random Forest':     RandomForestRegressor(
                             n_estimators=200, max_depth=8,
                             min_samples_leaf=3, random_state=42, n_jobs=-1),

    'Gradient Boosting': GradientBoostingRegressor(
                             n_estimators=300, max_depth=3,
                             learning_rate=0.05, random_state=42),

    'SVR (rbf)':         SVR(kernel='rbf',    C=5,   gamma='scale', epsilon=0.5),
    'SVR (linear)':      SVR(kernel='linear', C=1.0),
    'SVR (poly deg2)':   SVR(kernel='poly',   degree=2, C=1.0, gamma='scale'),

    'KNN (k=3)':         KNeighborsRegressor(n_neighbors=3),
    'KNN (k=5)':         KNeighborsRegressor(n_neighbors=5),
    'KNN (k=7)':         KNeighborsRegressor(n_neighbors=7),
    'KNN (k=9)':         KNeighborsRegressor(n_neighbors=9),
    'KNN (k=11)':        KNeighborsRegressor(n_neighbors=11),
}

print(f"   Modelos: {len(models)}")
print(f"   └─ Con normalización: {len([m for m in models if m in needs_scaling])}")
print(f"   └─ Sin normalización: {len([m for m in models if m not in needs_scaling])}")
print(f"   └─ Con límite de train: {len(TRAIN_LIMIT)} modelos")

# ==========================================
# 3. CONFIGURACIÓN DE LA VALIDACIÓN INTERNA
# ==========================================
print("\n3. CONFIGURING INTERNAL VALIDATION...")
print(f"   Estrategia: SINGLE TEMPORAL SPLIT (80% train / 20% validation)")
print(f"   Razón: pocas muestras → múltiples pliegues serían inestables.")
print(f"   La validación interna utiliza los últimos datos (más recientes),")
print(f"   respetando la estructura temporal (invierno → primavera).")
print(f"   ✅ Tras la validación, el modelo se reentrena con TODAS las")
print(f"      muestras del límite para que el paso 6 vea las curvas completas.")

# ==========================================
# 4. ENTRENAR Y EVALUAR (single temporal split)
# ==========================================
print("\n4. TRAINING MODELS...")
print("="*70)

results          = {}
val_scores_store = {}
trained_models   = {}
predictions      = {}

for name, model in models.items():
    X_tr, X_te = (X_train_norm, X_test_norm) if name in needs_scaling \
                 else (X_train_raw, X_test_raw)
    data_type = 'normalised' if name in needs_scaling else 'raw'

    # Aplicar límite de muestras
    if name in TRAIN_LIMIT:
        limit        = TRAIN_LIMIT[name]
        X_tr_limited = X_tr[:limit]
        y_tr_limited = y_train[:limit]
        print(f"\n   ▶ {name} ({data_type}) [límite={limit} muestras]...")
    else:
        X_tr_limited = X_tr
        y_tr_limited = y_train
        print(f"\n   ▶ {name} ({data_type}) [train={len(y_train)} muestras]...")

    n = len(y_tr_limited)

    # ── FASE 1: validación interna con split 80/20 ──────────────────
    if n >= 5:
        split_idx     = int(n * 0.8)
        X_train_split = X_tr_limited[:split_idx]
        y_train_split = y_tr_limited[:split_idx]
        X_val_split   = X_tr_limited[split_idx:]
        y_val_split   = y_tr_limited[split_idx:]

        model.fit(X_train_split, y_train_split)

        # ===== NUEVO: predicciones en train y val (además de test) =====
        y_pred_train = model.predict(X_train_split)
        y_pred_val   = model.predict(X_val_split)
        y_pred_test  = np.clip(model.predict(X_te), 0, y_train.max() + 10)

        # ---- Métricas RMSE (las que mandan) ----
        train_rmse = np.sqrt(mean_squared_error(y_train_split, y_pred_train))
        val_rmse   = np.sqrt(mean_squared_error(y_val_split, y_pred_val))
        test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred_test))

        # ---- Métricas secundarias ----
        train_r2 = r2_score(y_train_split, y_pred_train)
        val_r2   = r2_score(y_val_split, y_pred_val)
        test_r2  = r2_score(y_test, y_pred_test)
        mae      = mean_absolute_error(y_test, y_pred_test)

        # Guardar para el resumen de validaciones (opcional)
        val_scores_store[name] = [round(val_r2, 4)]

        print(f"      Split: {len(y_train_split)} train + {len(y_val_split)} val")
        print(f"      Train RMSE: {train_rmse:.2f} | Val RMSE: {val_rmse:.2f} | Gap: {val_rmse - train_rmse:.2f}")
        print(f"      Val R²: {val_r2:.4f}")

    else:
        # Caso con muy pocas muestras: no se hace validación interna
        model.fit(X_tr_limited, y_tr_limited)
        y_pred_test = np.clip(model.predict(X_te), 0, y_train.max() + 10)

        train_rmse = np.nan
        val_rmse   = np.nan
        test_rmse  = np.sqrt(mean_squared_error(y_test, y_pred_test))
        train_r2   = np.nan
        val_r2     = np.nan
        test_r2    = r2_score(y_test, y_pred_test)
        mae        = mean_absolute_error(y_test, y_pred_test)
        val_scores_store[name] = []
        print(f"      ⚠️  Solo {n} muestras → sin validación interna")

    # ── FASE 2: reentrenar con TODAS las muestras del límite ────────
    # Esto garantiza que all_models.pkl contiene el modelo completo
    # y que el paso 6 puede dibujar las learning curves hasta el límite real
    model.fit(X_tr_limited, y_tr_limited)
    print(f"      Reentrenado con {len(y_tr_limited)} muestras (modelo final guardado)")

    # Guardar todas las métricas (incluyendo las nuevas)
    results[name] = {
        'RMSE':        round(test_rmse, 2),     # Test RMSE
        'MAE':         round(mae, 2),
        'R2':          round(test_r2, 4),       # Test R²
        'Train_RMSE':  round(train_rmse, 2) if not np.isnan(train_rmse) else np.nan,
        'Val_RMSE':    round(val_rmse, 2) if not np.isnan(val_rmse) else np.nan,
        'Train_R2':    round(train_r2, 4) if not np.isnan(train_r2) else np.nan,
        'Val_R2':      val_r2,                  # puede ser np.nan
        'Val_std':     0.0,
        'Train_size':  len(y_train_split) if n >= 5 else len(y_tr_limited),
        'Val_size':    len(y_val_split)   if n >= 5 else 0,
        'Full_size':   len(y_tr_limited),
    }
    trained_models[name] = model   # ← modelo entrenado con todas las muestras
    predictions[name]    = y_pred_test

    print(f"      Test → RMSE: {test_rmse:.2f} | MAE: {mae:.2f} | R²: {test_r2:.4f}")

# ==========================================
# 5. FUNCIÓN DIAGNÓSTICO (BASADA 100% EN RMSE)
# ==========================================
def diagnose(row):
    train_rmse = row.get('Train_RMSE', np.nan)
    val_rmse   = row.get('Val_RMSE', np.nan)
    test_rmse  = row.get('RMSE', np.nan)

    # Si no hay validación, no podemos diagnosticar bien
    if pd.isna(train_rmse) or pd.isna(val_rmse):
        return '❓ SIN VALIDACIÓN'

    # 1. Calcular brechas (gap y gap_ratio)
    gap = val_rmse - train_rmse
    gap_ratio = gap / val_rmse if val_rmse > 0 else 0

    # =====================================================
    # DIAGNÓSTICO CON PRIORIDAD: UNDERFITTING PRIMERO
    # (porque si el error absoluto es enorme, da igual la brecha)
    # =====================================================

    # --- UNDERFITTING SEVERO ---
    if train_rmse > 1.5 * STD_Y and val_rmse > 1.5 * STD_Y:
        return '🔴 UNDERFITTING SEVERO'

    # --- UNDERFITTING (moderado) ---
    if train_rmse > 1.0 * STD_Y and val_rmse > 1.0 * STD_Y:
        return '🔴 UNDERFITTING'

    # --- OVERFITTING SEVERO (train ajusta perfecto, val se dispara) ---
    if train_rmse < 0.4 * STD_Y and gap_ratio > 0.5:
        return '🚨 OVERFITTING SEVERO'

    # --- OVERFITTING (brecha grande) ---
    # Umbrales dinámicos: gap_ratio > 0.35 o gap > 0.5*STD_Y
    if gap_ratio > 0.35 or gap > 0.5 * STD_Y:
        return '⚠️ OVERFITTING'

    # --- OVERFITTING LEVE ---
    # Umbrales dinámicos: gap_ratio > 0.20 o gap > 0.25*STD_Y
    if gap_ratio > 0.20 or gap > 0.25 * STD_Y:
        return '🟡 OVERFITTING LEVE'

    # --- Test RMSE alto aunque el split interno no lo detecte ---
    # (Por si el test es muy diferente al validation)
    if test_rmse > 1.2 * STD_Y:
        return '⚠️ MALO EN TEST (revisar datos)'

    # --- BUEN AJUSTE (el resto) ---
    return '✅ BUEN AJUSTE'

# ==========================================
# 6. RESUMEN Y RANKING CON DIAGNÓSTICO
# ==========================================
print("\n" + "="*70)
print("5. MODELS SUMMARY")
print("="*70)

df_summary = pd.DataFrame(results).T.sort_values('RMSE', ascending=True)
df_summary['Diagnosis'] = df_summary.apply(diagnose, axis=1)

print(f"\n   📊 Ranking completo (por RMSE ↑ — menor es mejor):")
print(df_summary.to_string())

best_model_name = df_summary.index[0]
best_metrics    = results[best_model_name]

print(f"\n   🏆 BEST MODEL: {best_model_name}")
print(f"   ┌──────────────────────────────────────────────┐")
print(f"   │ RMSE (test):    {best_metrics['RMSE']:.2f} personas              │")
print(f"   │ MAE:            {best_metrics['MAE']:.2f} personas              │")
print(f"   │ R² (test):      {best_metrics['R2']:.4f}                     │")
if not np.isnan(best_metrics['Val_R2']):
    print(f"   │ Val R² (80/20): {best_metrics['Val_R2']:.4f}                     │")
print(f"   │ Diagnosis:      {df_summary.loc[best_model_name, 'Diagnosis']}             │")
print(f"   └──────────────────────────────────────────────┘")

print(f"\n   📈 Comparativa RMSE (top 8):")
max_rmse = max(results[n]['RMSE'] for n in df_summary.index[:8])
for name in df_summary.index[:8]:
    rmse_val  = results[name]['RMSE']
    diagnosis = df_summary.loc[name, 'Diagnosis']
    bar       = '█' * max(0, int((1 - rmse_val / (max_rmse + 1)) * 40))
    print(f"   {name:<25} {bar} RMSE={rmse_val:.2f}  {diagnosis}")

# ==========================================
# 7. GUARDAR RESULTADOS
# ==========================================
print("\n6. SAVING RESULTS...")

with open(os.path.join(OUTPUT_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(trained_models[best_model_name], f)

residuals = y_test - predictions[best_model_name]
df_preds  = pd.DataFrame({
    'Actual':    y_test,
    'Predicted': np.round(predictions[best_model_name], 1),
    'Error':     np.round(residuals, 1),
    'Abs_Error': np.round(np.abs(residuals), 1),
    'Error_Pct': np.round(np.abs(residuals) / np.maximum(y_test, 1) * 100, 1)
})
df_preds.to_excel(os.path.join(OUTPUT_DIR, 'predictions.xlsx'), index=False)

df_summary.to_excel(os.path.join(OUTPUT_DIR, 'models_summary.xlsx'), index=True)

df_all = pd.DataFrame({'Actual': y_test})
for name, y_pred in predictions.items():
    df_all[name] = np.round(y_pred, 1)
df_all.to_excel(os.path.join(OUTPUT_DIR, 'all_predictions.xlsx'), index=False)

df_val = pd.DataFrame.from_dict(val_scores_store, orient='index')
if not df_val.empty and df_val.shape[1] > 0:
    df_val.columns = ['Val_R2']
df_val.to_excel(os.path.join(OUTPUT_DIR, 'val_scores_detail.xlsx'), index=True)

pd.DataFrame({'Occupancy': y_train}).to_excel(os.path.join(OUTPUT_DIR, 'y_train.xlsx'), index=False)
pd.DataFrame({'Occupancy': y_test}).to_excel(os.path.join(OUTPUT_DIR,  'y_test.xlsx'),  index=False)

config_info = {
    'classroom':       CLASSROOM,
    'task':            'regression',
    'ranking_by':      'RMSE',
    'n_models':        len(models),
    'val_method':      'Single temporal split 80/20 + retrain full',
    'n_train_samples': len(y_train),
    'n_test_samples':  len(y_test),
    'train_limits':    TRAIN_LIMIT,
    'best_model':      best_model_name,
    'best_rmse':       best_metrics['RMSE'],
    'best_mae':        best_metrics['MAE'],
    'best_r2':         best_metrics['R2'],
}
with open(os.path.join(OUTPUT_DIR, 'results.pkl'),     'wb') as f: pickle.dump(results,        f)
with open(os.path.join(OUTPUT_DIR, 'predictions.pkl'), 'wb') as f: pickle.dump(predictions,    f)
with open(os.path.join(OUTPUT_DIR, 'config.pkl'),      'wb') as f: pickle.dump(config_info,    f)
with open(os.path.join(OUTPUT_DIR, 'all_models.pkl'),  'wb') as f: pickle.dump(trained_models, f)

print(f"\n   📁 Archivos en '{OUTPUT_DIR}/':")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    icon = '📊' if fname.endswith('.xlsx') else '💾'
    print(f"   {icon} {fname:<40} ({size:.0f} KB)")

print("\n" + "="*70)
print("✅ Paso 5.1 completado!")
print("="*70)


AULA 1.6 - MACHINE LEARNING REGRESIÓN

1. LOADING DATA...
   Train: 37, Test: 25
   y_train: min=0, max=29, mean=12.24
   y_test:  min=0,  max=24,  mean=9.44
   STD_Y (referencia para diagnóstico): 10.08 personas

2. DEFINING MODELS...
   Modelos: 15
   └─ Con normalización: 12
   └─ Sin normalización: 3
   └─ Con límite de train: 2 modelos

3. CONFIGURING INTERNAL VALIDATION...
   Estrategia: SINGLE TEMPORAL SPLIT (80% train / 20% validation)
   Razón: pocas muestras → múltiples pliegues serían inestables.
   La validación interna utiliza los últimos datos (más recientes),
   respetando la estructura temporal (invierno → primavera).
   ✅ Tras la validación, el modelo se reentrena con TODAS las
      muestras del límite para que el paso 6 vea las curvas completas.

4. TRAINING MODELS...

   ▶ Linear Regression (normalised) [train=37 muestras]...
      Split: 29 train + 8 val
      Train RMSE: 3.45 | Val RMSE: 15.27 | Gap: 11.82
      Val R²: -1.1130
      Reentrenado con 37 muestras (